In [0]:
%pip install faker

In [0]:
dbutils.library.restartPython()

In [0]:
from faker import Faker
import random
import uuid
from datetime import datetime, timedelta

# Reproducibility: same seed = same fake data every time we regenerate.
# Real teams do this deliberately for testing pipelines against consistent data.
fake = Faker('en_IN')   # Indian locale - realistic names/addresses for our fintech context
Faker.seed(42)
random.seed(42)

CATALOG = "finpulse_dev"
VOLUME_PATH = "/Volumes/finpulse_dev/raw_landing/source_files"

In [0]:
import pandas as pd

# --- Branches ---
# Small, static reference table. 50 branches across Indian cities.
# No intentional messiness here — reference data is typically the cleanest
# source in a real bank, maintained manually by ops teams.

cities_states = [
    ("Mumbai", "Maharashtra"), ("Delhi", "Delhi"), ("Bangalore", "Karnataka"),
    ("Hyderabad", "Telangana"), ("Chennai", "Tamil Nadu"), ("Kolkata", "West Bengal"),
    ("Pune", "Maharashtra"), ("Ahmedabad", "Gujarat"), ("Jaipur", "Rajasthan"),
    ("Bhubaneswar", "Odisha"), ("Lucknow", "Uttar Pradesh"), ("Surat", "Gujarat"),
    ("Nagpur", "Maharashtra"), ("Indore", "Madhya Pradesh"), ("Bhopal", "Madhya Pradesh"),
    ("Visakhapatnam", "Andhra Pradesh"), ("Kochi", "Kerala"), ("Chandigarh", "Punjab"),
    ("Guwahati", "Assam"), ("Patna", "Bihar")
]

branch_types = ["METRO", "URBAN", "URBAN", "RURAL", "RURAL"]  # weighted toward urban

branches = []
for i in range(1, 51):
    city, state = cities_states[i % len(cities_states)]
    branches.append({
        "branch_id"   : f"BRN{i:03d}",
        "branch_name" : f"{city} Branch {i}",
        "city"        : city,
        "state"       : state,
        "branch_type" : random.choice(branch_types)
    })

branches_df = pd.DataFrame(branches)
print(f"Generated {len(branches_df)} branches")
branches_df.head()

In [0]:
branches_df.display()

==========**Below cell is giving error but claude told to skip below one**=========

In [0]:
# Write branches to the landing volume as CSV
# In a real bank, this would arrive as a file drop from an ops/reference data team.
# We write it once (no daily partitioning needed — branches rarely change).

# branches_output_path = f"{VOLUME_PATH}/branches/branches.csv"

# branches_df.to_csv(
#     f"/dbfs{branches_output_path}" if not VOLUME_PATH.startswith("/Volumes") 
#     else branches_output_path.replace("/Volumes", "/dbfs/Volumes"),
#     index=False
# )

# Verify it landed correctly
# import os
# verify_path = f"/dbfs/Volumes/finpulse_dev/raw_landing/source_files/branches/branches.csv"
# print(f"File exists: {os.path.exists(verify_path)}")
# print(f"File size: {os.path.getsize(verify_path)} bytes")

In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

# Convert pandas df to Spark df, write as CSV to Volume
branches_spark_df = spark.createDataFrame(branches_df)

branches_spark_df.coalesce(1).write.mode("overwrite").option("header", True).csv(
    f"{VOLUME_PATH}/branches/"
)

print("Branches written to Volume successfully")

# Verify
display(spark.read.option("header", True).csv(f"{VOLUME_PATH}/branches/"))

In [0]:
print(VOLUME_PATH)

In [0]:
print(len(branches_df))

In [0]:
# --- Customers ---
# 5,000 base records + ~1% duplicates injected = ~5,050 total rows
# Messiness injected:
#   1. ~1% duplicate records (slight name variation - simulates system migration issues)
#   2. ~5% null emails (some customers registered without email)
#   3. ~3% null phones
#   4. ~2% have PENDING/REJECTED KYC (rest are VERIFIED)

from datetime import date

kyc_statuses = ["VERIFIED"] * 90 + ["PENDING"] * 7 + ["REJECTED"] * 3  # weighted

customers = []
branch_ids = [f"BRN{i:03d}" for i in range(1, 51)]

for i in range(1, 5001):
    customer_id = f"CUST{i:05d}"
    first_name  = fake.first_name()
    last_name   = fake.last_name()
    dob         = fake.date_of_birth(minimum_age=18, maximum_age=75)
    
    # ~5% null email, ~3% null phone - realistic for legacy banking systems
    email = fake.email() if random.random() > 0.05 else None
    phone = fake.phone_number() if random.random() > 0.03 else None
    
    customers.append({
        "customer_id"          : customer_id,
        "first_name"           : first_name,
        "last_name"            : last_name,
        "date_of_birth"        : str(dob),
        "email"                : email,
        "phone"                : phone,
        "address_line"         : fake.street_address(),
        "city"                 : fake.city(),
        "state"                : fake.state(),
        "pincode"              : fake.postcode(),
        "kyc_status"           : random.choice(kyc_statuses),
        "customer_since_date"  : str(fake.date_between(start_date='-10y', end_date='-1y')),
        "last_updated_timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    })

# --- Inject duplicates (~1% = ~50 records) ---
# Simulates same customer appearing twice with slight name variation
# (common after system migrations or manual data entry errors)
duplicates = []
duplicate_pool = random.sample(customers, 50)

for original in duplicate_pool:
    dup = original.copy()
    # Slightly corrupt the name - real duplicate patterns in banking data
    dup["first_name"] = original["first_name"][:-1] if len(original["first_name"]) > 3 else original["first_name"] + "a"
    dup["last_updated_timestamp"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    duplicates.append(dup)

customers_df = pd.DataFrame(customers + duplicates)
customers_df = customers_df.sample(frac=1, random_state=42).reset_index(drop=True)  # shuffle

print(f"Total customer records : {len(customers_df)}")
print(f"Null emails            : {customers_df['email'].isna().sum()}")
print(f"Null phones            : {customers_df['phone'].isna().sum()}")
print(f"KYC status distribution:")
print(customers_df['kyc_status'].value_counts())

In [0]:
# Write customers to Volume as CSV - partitioned by date (simulating daily batch drop)
# Real core banking systems drop a full customer extract daily
# We write it as a single "today's" batch file for now

import datetime as dt

today = dt.date.today().strftime("%Y-%m-%d")

customers_spark_df = spark.createDataFrame(customers_df)

customers_spark_df.coalesce(1).write.mode("overwrite").option("header", True).csv(
    f"{VOLUME_PATH}/customers/load_date={today}/"
)

print(f"Customers written to: {VOLUME_PATH}/customers/load_date={today}/")

# Verify
display(
    spark.read.option("header", True).csv(
        f"{VOLUME_PATH}/customers/load_date={today}/"
    )
)

In [0]:
# --- Accounts ---
# ~7,500 accounts across 5,000 customers
# Some customers have 1 account, some have 2-3 (realistic distribution)
# Messiness injected:
#   1. ~2% orphan accounts referencing a non-existent branch (data entry errors)
#   2. ~3% DORMANT/CLOSED accounts (realistic portfolio mix)

account_types    = ["SAVINGS", "SAVINGS", "SAVINGS", "CURRENT", "LOAN"]  # weighted toward savings
account_statuses = ["ACTIVE"] * 92 + ["DORMANT"] * 5 + ["CLOSED"] * 3

customer_ids = [f"CUST{i:05d}" for i in range(1, 5001)]

accounts = []
account_counter = 1

for customer_id in customer_ids:
    # Distribute accounts realistically:
    # 60% of customers have 1 account, 30% have 2, 10% have 3
    num_accounts = random.choices([1, 2, 3], weights=[60, 30, 10])[0]
    
    for _ in range(num_accounts):
        account_id = f"ACC{account_counter:06d}"
        account_counter += 1

        # ~2% orphan: assign a non-existent branch_id
        # Simulates out-of-order loads or data entry errors in source system
        if random.random() < 0.02:
            branch_id = f"BRN{random.randint(51, 60):03d}"  # BRN051-060 don't exist
        else:
            branch_id = random.choice(branch_ids)

        accounts.append({
            "account_id"            : account_id,
            "customer_id"           : customer_id,
            "account_type"          : random.choice(account_types),
            "branch_id"             : branch_id,
            "open_date"             : str(fake.date_between(start_date='-8y', end_date='-1m')),
            "status"                : random.choice(account_statuses),
            "currency"              : "INR",
            "last_updated_timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        })

accounts_df = pd.DataFrame(accounts)

# Quick sanity checks
orphan_count = accounts_df[~accounts_df['branch_id'].isin([f"BRN{i:03d}" for i in range(1, 51)])].shape[0]

print(f"Total accounts generated : {len(accounts_df)}")
print(f"Orphan branch references : {orphan_count}")
print(f"Account type distribution:")
print(accounts_df['account_type'].value_counts())
print(f"\nAccount status distribution:")
print(accounts_df['status'].value_counts())

In [0]:
# Write accounts to Volume - daily batch, date-partitioned (same pattern as customers)
accounts_spark_df = spark.createDataFrame(accounts_df)

accounts_spark_df.coalesce(1).write.mode("overwrite").option("header", True).csv(
    f"{VOLUME_PATH}/accounts/load_date={today}/"
)

print(f"Accounts written to: {VOLUME_PATH}/accounts/load_date={today}/")

# Verify row count from file matches what we generated
loaded_count = spark.read.option("header", True).csv(
    f"{VOLUME_PATH}/accounts/load_date={today}/"
).count()

print(f"Rows written and verified: {loaded_count}")

In [0]:
# --- Cards ---
# ~6,000 cards across ~7,471 accounts
# Not every account has a card (LOAN accounts typically don't)
# Messiness injected:
#   1. ~2% BLOCKED/EXPIRED cards (realistic portfolio)
#   2. ~5% cards belonging to LOAN accounts (technically shouldn't happen - data quality issue)

card_networks = ["VISA", "VISA", "MASTERCARD", "RUPAY", "RUPAY"]
card_statuses = ["ACTIVE"] * 92 + ["BLOCKED"] * 5 + ["EXPIRED"] * 3

savings_current_ids = accounts_df[
    accounts_df['account_type'].isin(['SAVINGS', 'CURRENT'])
]['account_id'].tolist()

loan_ids = accounts_df[
    accounts_df['account_type'] == 'LOAN'
]['account_id'].tolist()

card_account_ids = (
    random.sample(savings_current_ids, min(5850, len(savings_current_ids))) +
    random.sample(loan_ids, min(150, len(loan_ids)))
)

cards = []
for i, account_id in enumerate(card_account_ids, 1):
    account_type = accounts_df[
        accounts_df['account_id'] == account_id
    ]['account_type'].values[0]

    card_type  = "CREDIT" if account_type == "CURRENT" else "DEBIT"
    issue_date = fake.date_between(start_date='-5y', end_date='-1m')

    try:
        expiry_date = issue_date.replace(year=issue_date.year + 5)
    except ValueError:
        # Handles Feb 29 on leap years - fall back to Feb 28
        expiry_date = issue_date.replace(month=2, day=28, year=issue_date.year + 5)

    cards.append({
        "card_id"     : f"CRD{i:06d}",
        "account_id"  : account_id,
        "card_type"   : card_type,
        "card_network": random.choice(card_networks),
        "issue_date"  : str(issue_date),
        "expiry_date" : str(expiry_date),
        "status"      : random.choice(card_statuses)
    })

cards_df = pd.DataFrame(cards)

# Sanity checks
loan_cards = cards_df[cards_df['account_id'].isin(loan_ids)]

print(f"Total cards generated     : {len(cards_df)}")
print(f"Cards on LOAN accounts    : {len(loan_cards)}  <- intentional anomaly")
print(f"\nCard type distribution:")
print(cards_df['card_type'].value_counts())
print(f"\nCard network distribution:")
print(cards_df['card_network'].value_counts())
print(f"\nCard status distribution:")
print(cards_df['status'].value_counts())

In [0]:
# Write cards to Volume - daily batch, date-partitioned
cards_spark_df = spark.createDataFrame(cards_df)

cards_spark_df.coalesce(1).write.mode("overwrite").option("header", True).csv(
    f"{VOLUME_PATH}/cards/load_date={today}/"
)

loaded_count = spark.read.option("header", True).csv(
    f"{VOLUME_PATH}/cards/load_date={today}/"
).count()

print(f"Cards written and verified: {loaded_count}")

In [0]:
# --- Transactions ---
# ~200,000 transactions across 180 days (6 months)
# ~1,100 transactions per day
# Messiness injected:
#   1. ~2% late-arriving records (transaction_timestamp behind file date by 1-3 days)
#   2. ~1% duplicate transaction_ids (processor retry simulation)
#   3. ~1% invalid amounts (zero or negative)
#   4. ~1% invalid currency codes

from datetime import timedelta
import hashlib

channels  = ["ATM", "POS", "ONLINE", "UPI", "BRANCH"]
txn_types = ["DEBIT", "DEBIT", "DEBIT", "CREDIT", "TRANSFER"]  # debits dominate realistically
statuses  = ["SUCCESS"] * 92 + ["FAILED"] * 5 + ["PENDING"] * 3
currencies = ["INR"] * 97 + ["USD", "EUR", "GBP"]  # ~3% invalid/foreign currency

merchants = [
    "Amazon India", "Flipkart", "Swiggy", "Zomato", "BigBasket",
    "Uber", "Ola", "MakeMyTrip", "Myntra", "Reliance Digital",
    "DMart", "IRCTC", "BookMyShow", "PhonePe", "Paytm Mall"
]

# Valid account IDs pool - only use ACTIVE accounts for transactions
active_account_ids = accounts_df[
    accounts_df['status'] == 'ACTIVE'
]['account_id'].tolist()

# Date range: 6 months back from today
start_date = dt.date.today() - timedelta(days=180)
end_date   = dt.date.today() - timedelta(days=1)

print(f"Active accounts available : {len(active_account_ids)}")
print(f"Transaction date range    : {start_date} to {end_date}")
print(f"Days to generate          : {(end_date - start_date).days + 1}")

In [0]:
# Part 2: Day-by-day transaction generation
all_transaction_ids = []
daily_stats = []

current_date = start_date

while current_date <= end_date:
    daily_transactions = []
    date_str = current_date.strftime("%Y-%m-%d")

    is_weekend  = current_date.weekday() >= 5
    daily_count = random.randint(800, 950) if is_weekend else random.randint(1050, 1200)

    for _ in range(daily_count):
        txn_id  = f"TXN{uuid.uuid4().hex[:12].upper()}"
        account = random.choice(active_account_ids)

        if random.random() < 0.02:
            actual_date = current_date - timedelta(days=random.randint(1, 3))
        else:
            actual_date = current_date

        txn_hour   = random.randint(6, 23)
        txn_minute = random.randint(0, 59)
        txn_second = random.randint(0, 59)
        txn_timestamp = datetime.combine(actual_date,
                            datetime.min.time().replace(
                                hour=txn_hour,
                                minute=txn_minute,
                                second=txn_second
                            ))

        channel = random.choice(channels)

        if random.random() < 0.01:
            amount = round(random.uniform(-500, 0), 2)
        else:
            amount = round(random.uniform(10, 50000), 2)

        daily_transactions.append({
            "transaction_id"       : txn_id,
            "account_id"           : account,
            "transaction_timestamp": str(txn_timestamp),
            "amount"               : amount,
            "currency"             : random.choice(currencies),
            "transaction_type"     : random.choice(txn_types),
            "channel"              : channel,
            "merchant_name"        : random.choice(merchants) if channel in ["POS", "ONLINE"] else None,
            "status"               : random.choice(statuses),
            "ingestion_timestamp"  : str(datetime.combine(current_date, datetime.min.time()))
        })

        all_transaction_ids.append(txn_id)

    # ~1% duplicates: randomly re-inject from earlier days
    previous_ids = all_transaction_ids[:-daily_count]
    if len(previous_ids) > 10 and random.random() < 0.4:
        num_dups = random.randint(1, min(5, len(previous_ids)))
        dup_ids  = random.sample(previous_ids, num_dups)
        for dup_id in dup_ids:
            dup = next((t for t in daily_transactions if t["transaction_id"] == dup_id), None)
            if dup is None:
                dup = daily_transactions[0].copy()
                dup["transaction_id"] = dup_id
            daily_transactions.append(dup)

    daily_df = spark.createDataFrame(pd.DataFrame(daily_transactions))

    daily_df.coalesce(1).write.mode("overwrite").option("header", True).csv(
        f"{VOLUME_PATH}/transactions/transaction_date={date_str}/"
    )

    daily_stats.append({
        "date"      : date_str,
        "records"   : len(daily_transactions),
        "is_weekend": is_weekend
    })

    current_date += timedelta(days=1)

print(f"Transaction generation complete!")
print(f"Total daily files written : {len(daily_stats)}")
print(f"Total records generated   : {sum(s['records'] for s in daily_stats)}")
print(f"Avg weekday volume        : {int(sum(s['records'] for s in daily_stats if not s['is_weekend']) / sum(1 for s in daily_stats if not s['is_weekend']))}")
print(f"Avg weekend volume        : {int(sum(s['records'] for s in daily_stats if s['is_weekend']) / sum(1 for s in daily_stats if s['is_weekend']))}")

In [0]:
from pyspark.sql.functions import col, to_timestamp

# Read all 180 daily files back as one DataFrame
transactions_all = spark.read.option("header", True).csv(
    f"{VOLUME_PATH}/transactions/"
)

# Cast columns to correct types before filtering
transactions_typed = transactions_all \
    .withColumn("amount", col("amount").cast("double")) \
    .withColumn("transaction_timestamp", col("transaction_timestamp").cast("timestamp")) \
    .withColumn("ingestion_timestamp", col("ingestion_timestamp").cast("timestamp"))

total            = transactions_typed.count()
late_arriving    = transactions_typed.filter(
    col("transaction_timestamp") < col("ingestion_timestamp")
).count()
invalid_amounts  = transactions_typed.filter(
    col("amount") <= 0
).count()
duplicate_ids    = total - transactions_typed.select("transaction_id").distinct().count()
foreign_currency = transactions_typed.filter(
    ~col("currency").isin(["INR"])
).count()

print(f"Total records read back   : {total}")
print(f"Late arriving records     : {late_arriving} ({round(late_arriving/total*100, 2)}%)")
print(f"Invalid amounts           : {invalid_amounts} ({round(invalid_amounts/total*100, 2)}%)")
print(f"Duplicate transaction IDs : {duplicate_ids}")
print(f"Foreign currency records  : {foreign_currency} ({round(foreign_currency/total*100, 2)}%)")